# Part 4

In [ ]:
import os
from pathlib import Path
import mysql.connector
import pandas as pd

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]
data_folder = Path("data")

# Connect to the cloud stock database
aws_connection = mysql.connector.connect(
    host=os.environ["AWS_DB_HOST"],
    user=os.environ["AWS_DB_USER"],
    password=os.environ["AWS_DB_PASSWORD"],
    port=3306,
    database="stock_attention",
    connection_timeout=20
)

# Connect to the local tweet database and warehouse
local_connection = mysql.connector.connect(
    host=os.environ.get("LOCAL_DB_HOST", "localhost"),
    user=os.environ["LOCAL_DB_USER"],
    password=os.environ["LOCAL_DB_PASSWORD"],
    port=3306,
    database="stock_attention",
    connection_timeout=20
)

aws_cursor = aws_connection.cursor(dictionary=True)
local_cursor = local_connection.cursor(dictionary=True)

local_cursor.execute("SET SESSION time_zone = '+00:00'")

print("Connected to cloud and local databases")

### Extract stock from AWS

In [ ]:
stock_parts = []

for ticker in tickers:
    aws_cursor.execute(f"""
        SELECT
            stock_date,
            open_price,
            high_price,
            low_price,
            close_price,
            adjusted_close,
            volume
        FROM `{ticker}`
        WHERE stock_date >= '2020-01-01'
          AND stock_date < '2023-01-01'
        ORDER BY stock_date
    """)

    stock = pd.DataFrame(aws_cursor.fetchall())
    stock["ticker"] = ticker
    stock_parts.append(stock)

stocks = pd.concat(stock_parts, ignore_index=True)
stocks["stock_date"] = pd.to_datetime(stocks["stock_date"])

if stocks.duplicated(["ticker", "stock_date"]).any():
    raise ValueError("Duplicate stock records found")

print("Stock rows extracted:", len(stocks))
print(stocks.groupby("ticker").size())

## Extract daily tweet totals from local SQL

In [ ]:
# Extract daily tweet totals from local MySQL

local_cursor.execute("""
    SELECT
        s.ticker,
        DATE(FROM_UNIXTIME(t.unix_time)) AS stock_date,
        COUNT(*) AS tweet_count,
        SUM(count_word(t.tweet_text, 'buy')) AS buy_count,
        SUM(count_word(t.tweet_text, 'sell')) AS sell_count
    FROM selected_tickers AS s
    JOIN tweets AS t
        ON UPPER(t.tweet_text) REGEXP
           CONCAT(
               '(^|[^A-Z0-9_])',
               s.ticker,
               '([^A-Z0-9_]|$)'
           )
    WHERE t.unix_time >= 1577836800
      AND t.unix_time < 1672531200
    GROUP BY
        s.ticker,
        DATE(FROM_UNIXTIME(t.unix_time))
""")

daily_tweets = pd.DataFrame(
    local_cursor.fetchall(),
    columns=[
        "ticker", "stock_date",
        "tweet_count", "buy_count", "sell_count"
    ]
)

daily_tweets["stock_date"] = pd.to_datetime(
    daily_tweets["stock_date"]
)

print("Daily tweet groups extracted:", len(daily_tweets))
print(daily_tweets.head())

In [ ]:
# Read combined daily Google Trends CSV

trends = pd.read_csv(
    "data_folder / "trends_daily.csv",
    parse_dates=["Date"]
)

trends = trends.rename(columns={
    "Date": "stock_date",
    "Ticker": "ticker",
    "Interest": "google_interest",
    "Source_File": "trends_source_file"
})

trends = trends[
    [
        "ticker",
        "stock_date",
        "google_interest",
        "trends_source_file"
    ]
]

if trends.duplicated(["ticker", "stock_date"]).any():
    raise ValueError("Duplicate Google Trends records found")

print("Google Trends rows read:", len(trends))

In [ ]:
# Combine the 3 sources
warehouse_data = stocks.merge(
    daily_tweets,
    on=["ticker", "stock_date"],
    how="left",
    validate="one_to_one"
)

warehouse_data = warehouse_data.merge(
    trends,
    on=["ticker", "stock_date"],
    how="left",
    validate="one_to_one"
)

# No matching tweets in our dataset means a count of zero.
count_columns = ["tweet_count", "buy_count", "sell_count"]

warehouse_data[count_columns] = (
    warehouse_data[count_columns]
    .fillna(0)
    .astype("int64")
)

# Missing search observations must not silently become zero.
if warehouse_data[
    ["google_interest", "trends_source_file"]
].isna().any().any():
    raise ValueError("Google Trends data missing for a trading day")

if len(warehouse_data) != len(stocks):
    raise ValueError("Joining changed the number of stock records")

print("Combined fact rows:", len(warehouse_data))
print(warehouse_data.head().to_string(index=False))

In [ ]:
# Populate the date dimension

date_rows = []

for day in pd.date_range("2020-01-01", "2022-12-31"):
    date_rows.append((
        day.date(),
        day.year,
        day.month,
        day.day,
        day.isoweekday(),
        int(day.dayofweek >= 5)
    ))

local_cursor.executemany("""
    INSERT INTO dim_date (
        stock_date,
        calendar_year,
        calendar_month,
        day_of_month,
        day_of_week,
        is_weekend
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        calendar_year = VALUES(calendar_year),
        calendar_month = VALUES(calendar_month),
        day_of_month = VALUES(day_of_month),
        day_of_week = VALUES(day_of_week),
        is_weekend = VALUES(is_weekend)
""", date_rows)

local_connection.commit()

print("Date dimension loaded:", len(date_rows), "dates")

In [ ]:
# Load the fact table

fact_rows = []

for _, row in warehouse_data.iterrows():
    fact_rows.append((
        row["ticker"],
        row["stock_date"].date(),
        float(row["open_price"]),
        float(row["high_price"]),
        float(row["low_price"]),
        float(row["close_price"]),
        float(row["adjusted_close"]),
        int(row["volume"]),
        int(row["tweet_count"]),
        int(row["buy_count"]),
        int(row["sell_count"]),
        float(row["google_interest"]),
        row["trends_source_file"]
    ))

local_cursor.executemany("""
    INSERT INTO fact_daily (
        ticker,
        stock_date,
        open_price,
        high_price,
        low_price,
        close_price,
        adjusted_close,
        volume,
        tweet_count,
        buy_count,
        sell_count,
        google_interest,
        trends_source_file
    )
    VALUES (
        %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s
    )
    ON DUPLICATE KEY UPDATE
        open_price = VALUES(open_price),
        high_price = VALUES(high_price),
        low_price = VALUES(low_price),
        close_price = VALUES(close_price),
        adjusted_close = VALUES(adjusted_close),
        volume = VALUES(volume),
        tweet_count = VALUES(tweet_count),
        buy_count = VALUES(buy_count),
        sell_count = VALUES(sell_count),
        google_interest = VALUES(google_interest),
        trends_source_file = VALUES(trends_source_file)
""", fact_rows)

local_connection.commit()

print("Fact table loaded:", len(fact_rows), "records")


In [ ]:
# Close connection once check in local SQL
aws_cursor.close()
local_cursor.close()
aws_connection.close()
local_connection.close()

In [ ]:
#Check and read the reference file

from pathlib import Path

file = Path("data_folder / "trends_reference_2020_2022.csv")

with file.open(encoding="utf-8-sig") as csv_file:
    for _ in range(8):
        print(csv_file.readline().rstrip())

In [ ]:
# Change daily file to weekly reference
from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

# Read the daily data already created for Part 3.
daily = pd.read_csv(
    "data_folder / "trends_daily.csv",
    parse_dates=["Date"]
)

daily["Quarter"] = daily["Date"].dt.to_period("Q").astype(str)

# Read the weekly reference, skipping Google's introductory lines.
reference_file = Path("data_folder / "trends_reference_2020_2022.csv")
lines = reference_file.read_text(
    encoding="utf-8-sig"
).splitlines()

header_row = next(
    i for i, line in enumerate(lines)
    if line.split(",")[0].strip().strip('"') == "Week"
)

reference = pd.read_csv(
    StringIO("\n".join(lines[header_row:]))
)

reference["Week"] = pd.to_datetime(reference["Week"])

weekly_parts = []

for column in reference.columns:
    ticker = column.split(":")[0].strip()

    if ticker in tickers:
        values = (
            reference[column]
            .astype(str)
            .str.strip()
            .replace("<1", "0.5")
        )

        weekly_parts.append(pd.DataFrame({
            "Ticker": ticker,
            "Week": reference["Week"],
            "Reference_Interest": pd.to_numeric(
                values, errors="raise"
            )
        }))

weekly = pd.concat(weekly_parts, ignore_index=True)

# Match every daily observation to its reference week.
matched = pd.merge_asof(
    daily.sort_values("Date"),
    weekly.sort_values("Week"),
    left_on="Date",
    right_on="Week",
    by="Ticker",
    direction="backward",
    tolerance=pd.Timedelta(days=6)
)

if matched["Reference_Interest"].isna().any():
    raise ValueError("Some daily dates have no reference week")

# Calculate daily-file averages for each reference week.
weekly_pairs = matched.groupby(
    ["Ticker", "Quarter", "Week"],
    as_index=False
).agg(
    Daily_Mean=("Interest", "mean"),
    Reference_Interest=("Reference_Interest", "first"),
    Days=("Date", "count")
)

# Use only complete weeks within a single quarter.
weekly_pairs = weekly_pairs[weekly_pairs["Days"] == 7]

scale_rows = []

for (ticker, quarter), group in weekly_pairs.groupby(
    ["Ticker", "Quarter"]
):
    x = group["Daily_Mean"].to_numpy(dtype=float)
    y = group["Reference_Interest"].to_numpy(dtype=float)

    # Fit reference interest ≈ scale × daily-file interest.
    denominator = np.dot(x, x)

    if denominator == 0:
        raise ValueError(f"Cannot calibrate {ticker}, {quarter}")

    scale = np.dot(x, y) / denominator

    if scale <= 0:
        raise ValueError(f"Invalid scale for {ticker}, {quarter}")

    scale_rows.append({
        "Ticker": ticker,
        "Quarter": quarter,
        "Scale_Factor": scale,
        "Weeks_Used": len(group),
        "Fit_RMSE": np.sqrt(np.mean((scale * x - y) ** 2))
    })

scales = pd.DataFrame(scale_rows)

if len(scales) != 60:
    raise ValueError("Expected 5 tickers × 12 quarters = 60 factors")

daily = daily.merge(
    scales[["Ticker", "Quarter", "Scale_Factor"]],
    on=["Ticker", "Quarter"],
    how="left",
    validate="many_to_one"
)

if daily["Scale_Factor"].isna().any():
    raise ValueError("Missing scale factors")

daily["Calibrated_Interest"] = (
    daily["Interest"] * daily["Scale_Factor"]
)

daily.to_csv(
    "data_folder / "trends_daily_calibrated.csv",
    index=False,
    date_format="%Y-%m-%d"
)

scales.to_csv(
    "data_folder / "trends_scale_factors.csv",
    index=False
)

print("Calibrated daily rows:", len(daily))
print("Ticker-quarter scale factors:", len(scales))
print(scales.head(10).to_string(index=False))



#Fit_RMSE shows how closely the scaled daily averages match the weekly reference 
# Lower values indicate closer agreement

In [ ]:
# Update matches only the trading days already in fact table

import mysql.connector
import pandas as pd

calibrated = pd.read_csv(
    "data_folder / "trends_daily_calibrated.csv",
    parse_dates=["Date"]
)

local_connection = mysql.connector.connect(
    host=os.environ.get("LOCAL_DB_HOST", "localhost"),
    user=os.environ["LOCAL_DB_USER"],
    password=os.environ["LOCAL_DB_PASSWORD"],
    port=3306,
    database="stock_attention"
)

cursor = local_connection.cursor()

update_rows = [
    (
        float(row.Calibrated_Interest),
        row.Ticker,
        row.Date.date()
    )
    for row in calibrated.itertuples(index=False)
]

cursor.executemany("""
    UPDATE fact_daily
    SET google_interest_calibrated = %s
    WHERE ticker = %s
      AND stock_date = %s
""", update_rows)

local_connection.commit()

cursor.execute("""
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(google_interest_calibrated) AS calibrated_rows
    FROM fact_daily
""")

print("Fact rows, calibrated rows:", cursor.fetchone())

cursor.close()
local_connection.close()